# Triball: a benchmark for rigid bodies in contact.

This test simulates a rigid body that is in contact with the ground plane and verifies the contact forces experienced at each contact point.

The model consists of three solid spheres connected by rods in a triangular configuration. This benchmark varies both triball configurations as well as the velocity of the center of mass(com). 
The model doesn't consist of any joints.

Two scenarios have been chosen to validate the simulated solutions and to represent simple and complex scenarios.

In **simple** scenario the triball model is at rest and no external is applied except 
gravitational and contact forces, here the test parameter being the configuration in which balls are arranged (eg: equilateral or isosceles triangle).
The normal/contact force experience at the contact point is a function of triball configuration, which forces can be computed analytically 
and can be used to compare against the simulated/ numerical solution.

In **complex** scenario the triball model has been given a specific initial velocity (both linear and angular).

## Simple scenario

As the rigid body is at rest, it should not experience any friction force. The forces experienced by it are a normal force in the upward direction and gravitational force in the downward direction. 
Which should be equal to each other in magnitude. In an equilibrium state, any rigid body should have a zero net force and torque about any point.

![default_gzclient_camera(1)-2024-06-26T23_57_14 798017](https://github.com/yaswanth1701/simulation_benchmark/assets/92177410/370f8aeb-b7de-42d7-a169-cbc3ba2a48d4)

## Complex scenario
As the rigid body is given a specific initial velocity, friction force will be acting in the horizontal plane. 

## Definition of coordinate frames and variable

![triball_diagram](https://github.com/yaswanth1701/simulation_benchmark/assets/92177410/1b2589e8-efb3-4885-801f-0d9912178565)

Consider an inertial frame $O$ and a rigid body with a coordinate frame $c$ attached at the center of mass:

- The position of the center of gravity (cg) in $O$  is given by  **c**.
- The orientation of $c$ with respect to $O$ is given by the quaternion **$q$**.
- A rotation matrix from $O$ to $c$ : $R(q)$ 
- Angular velocity in frame $c$ : $\omega$
- Three contact points for the model are expressed in frame $O$ as $C_a$, $C_b$ and $C_c$.
- The three contact force are denoted as $N_a$, $N_b$ and $N_c$ and are expressed in the $O$ frame
  
Rigid body has the following inertial parameters:
- Total mass *m*
- inertia matrix $I$ expressed in $c$
- Density of model be $\rho$
- Radius of ball/sphere is denoted as $R_s=0.02$.
- Radius of rod/cylinder is denoted as $R_c=0.005$.
- Mass of each ball and rod is $m_s$ and $m_c$ respectively.

### Solution for simple scenario:

In [32]:
import matplotlib.pyplot as np
import numpy as np
import sympy as sym

### Geometric variables:

- For simplifying the configuration of balls, we assume the coordinates of
  <br> ball $A$ (shown in the above figure) and the altitude of the triangle 
  $AD$ <br> are fixed throughout the test cases.
- The variables are angles $\angle BAD$ ($\theta _1$) and $\angle CAD$ ($\theta _2$).
- The height of the altitude be $h$ and the length from origin $O$ to point D be $k$.
- Then coordinate of ball $A$ will be $P_a(h - k, 0, 0.02)$
- Coordinates of ball $B$ will be $P_b(-k, h*tan\theta _1, 0.02)$.
- Coordinates of ball $C$ will be $P_c(-k, -h*tan\theta _2, 0.02)$.
- Further coordinates of center of mass (com) of each rod is denoted by $P_{ab}$, $P_{bc}$, $P_{ca}$.


In [33]:
rho = 200
# total ten pair of angles theta_1 and theta_2
N = 10

#altittude height (m)
h = 0.15
k = 0.05

theta_1 = np.linspace(np.pi/6, np.pi/2, N, endpoint=False)
theta_2 = np.linspace(np.pi/6, np.pi/3, N)

# coordinates of ball A
p_a = np.repeat(np.array([[h-k, 0, 0.02]]), N, axis=0)

# coordinates of ball B 
p_b_x = np.ones((N,1)) * -k
p_b_y = np.tan(theta_1).reshape(N,1)
p_b_z = np.ones((N,1)) * 0.02
p_b = np.hstack((p_b_x, p_b_y, p_b_z))

# coordinates of ball C
p_c_x = np.ones((N,1)) * -k
p_c_y = - np.tan(theta_2).reshape(N,1)
p_c_z = np.ones((N,1)) * 0.02
p_c = np.hstack((p_c_x, p_c_y, p_c_z))

# com of rod AB
p_ab = (p_a + p_b)/2

# com of rod BC
p_bc = (p_b + p_c)/2

# com of rod CA
p_ca = (p_c + p_a)/2

As the body is at rest the net force and moment about any point should be zeros. We choose to calculate the net moment about point $P_a$.

- Force balance gives:
  
   $N_a = [0, 0, n_a]^T, \text{ } N_b = [0, 0, n_b]^T \text{ and } N_c = [0, 0, n_c]^T$ 
    
   $g = [0, 0, -9.8]^T$ 
    
   $F_{net}= ma$
  
   $a = 0$
  
   $N_a + N_b + N_c + g = 0$ -> $n_a + n_b + n_c = 9.8$ ---- $equation1$
   
   $\tau _a = 0$
  
   $m_c*g \times (P_{ab} - P_a)$ + $(m_s *g + N_b) \times (P_b - P_a)$ +
   $m_c *g \times (P_{bc} - P_a)$ +

   $(m_s *g + N_c) \times (P_c - P_a)$ + $m_c *g \times (P_{ca} - P_a)$ = 0 ----- $equation2 \text{ and } 3$
   
So, we get a total of three linear equations to solve for $n_a$, $n_b$, and $n_c$.

In [ ]:
def compute_contact_forces():
    # creating symbolic variables for normal forces
    n_a = sym.symbols("n_a")
    n_b = sym.symbols("n_b")
    n_c = sym.symbols("n_c")

